In [ ]:
"""
=============================================================
STEP 1: DATA COLLECTION — Synthetic Document Generator
=============================================================
Since we can't scrape real corporate docs, we generate realistic
samples: Invoices, Contracts, and Reports — both structured and
unstructured — saved as PNG images simulating scanned documents.
"""

import os
import random
import textwrap
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import numpy as np

# ─── Output directory ─────────────────────────────────────────────────────────
SAMPLES_DIR = os.path.join(os.path.dirname(__file__), '..', 'data', 'samples')
os.makedirs(SAMPLES_DIR, exist_ok=True)

# ─── Helper: get a basic PIL font (fallback to default if no truetype) ─────────
def get_font(size=14):
    try:
        return ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", size)
    except:
        return ImageFont.load_default()

def get_bold_font(size=16):
    try:
        return ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", size)
    except:
        return ImageFont.load_default()


# ─── INVOICE GENERATOR ────────────────────────────────────────────────────────
def generate_invoice(idx=1):
    """
    Creates a structured Invoice image with:
    - Company header
    - Invoice number, date, due date
    - Line items table
    - Subtotal / Tax / Total
    """
    W, H = 794, 1123  # A4 at 96 dpi
    img = Image.new('RGB', (W, H), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)

    # Fonts
    title_font  = get_bold_font(28)
    header_font = get_bold_font(14)
    body_font   = get_font(12)
    small_font  = get_font(10)

    # Random data
    companies = ["Apex Solutions Ltd.", "NovaTech Corp.", "GlobalPrime Inc.", "DataBridge LLC"]
    clients   = ["TechVentures Pvt Ltd", "Horizon Systems", "CloudBase Inc.", "NetFusion Corp"]
    company   = random.choice(companies)
    client    = random.choice(clients)
    inv_num   = f"INV-{random.randint(10000,99999)}"
    inv_date  = f"{random.randint(1,28):02d}/{random.randint(1,12):02d}/2024"
    due_date  = f"{random.randint(1,28):02d}/{random.randint(1,12):02d}/2024"

    items = [
        ("Software Development Services", random.randint(10,80), random.randint(50,200)),
        ("Cloud Hosting (Monthly)",       random.randint(1, 12), random.randint(100,500)),
        ("Technical Support",             random.randint(5, 30), random.randint(40,120)),
        ("Data Analytics Module",         random.randint(1,  5), random.randint(500,2000)),
    ]
    num_items = random.randint(2, 4)
    items = random.sample(items, num_items)

    # ── Draw header bar ──
    draw.rectangle([0, 0, W, 90], fill=(30, 60, 120))
    draw.text((40, 20), company, font=title_font, fill=(255, 255, 255))
    draw.text((40, 60), "123 Corporate Park, Business City, IN-411001", font=small_font, fill=(200, 220, 255))

    # ── INVOICE title ──
    draw.text((W-200, 20), "INVOICE", font=title_font, fill=(255, 200, 0))

    # ── Invoice meta box ──
    draw.rectangle([W-260, 100, W-40, 200], outline=(30,60,120), width=2)
    draw.text((W-250, 110), f"Invoice #: {inv_num}",  font=header_font, fill=(30,60,120))
    draw.text((W-250, 135), f"Date:      {inv_date}", font=body_font,   fill=(50,50,50))
    draw.text((W-250, 155), f"Due Date:  {due_date}", font=body_font,   fill=(50,50,50))

    # ── Bill To ──
    draw.text((40, 110), "BILL TO:", font=header_font, fill=(30,60,120))
    draw.text((40, 135), client,     font=body_font,   fill=(50,50,50))
    draw.text((40, 155), "456 Client Avenue, Tech City",  font=body_font, fill=(80,80,80))
    draw.text((40, 172), "GST: 27ABCDE1234F1Z5",          font=small_font, fill=(100,100,100))

    # ── Table header ──
    y = 220
    draw.rectangle([40, y, W-40, y+30], fill=(30,60,120))
    cols = [40, 320, 450, 560, 680]
    headers = ["Description", "Qty", "Unit Price", "Amount"]
    for i, h in enumerate(headers):
        draw.text((cols[i]+5, y+7), h, font=header_font, fill=(255,255,255))

    # ── Table rows ──
    y += 30
    subtotal = 0
    for i, (desc, qty, price) in enumerate(items):
        amount = qty * price
        subtotal += amount
        bg = (240, 245, 255) if i % 2 == 0 else (255, 255, 255)
        draw.rectangle([40, y, W-40, y+28], fill=bg)
        draw.text((cols[0]+5, y+6), desc,            font=body_font, fill=(40,40,40))
        draw.text((cols[1]+5, y+6), str(qty),         font=body_font, fill=(40,40,40))
        draw.text((cols[2]+5, y+6), f"${price:,.2f}", font=body_font, fill=(40,40,40))
        draw.text((cols[3]+5, y+6), f"${amount:,.2f}",font=body_font, fill=(40,40,40))
        y += 28

    # ── Totals ──
    tax  = subtotal * 0.18
    total= subtotal + tax
    y += 20
    draw.line([400, y, W-40, y], fill=(200,200,200), width=1)
    y += 10
    draw.text((cols[2]+5, y),    "Subtotal:", font=body_font,   fill=(60,60,60))
    draw.text((cols[3]+5, y),    f"${subtotal:,.2f}", font=body_font, fill=(60,60,60))
    draw.text((cols[2]+5, y+22), "GST (18%):",font=body_font,   fill=(60,60,60))
    draw.text((cols[3]+5, y+22), f"${tax:,.2f}",      font=body_font, fill=(60,60,60))
    draw.rectangle([400, y+46, W-40, y+76], fill=(30,60,120))
    draw.text((cols[2]+5, y+54), "TOTAL:",    font=header_font, fill=(255,255,255))
    draw.text((cols[3]+5, y+54), f"${total:,.2f}", font=header_font, fill=(255,200,0))

    # ── Notes ──
    y += 120
    draw.text((40, y), "Payment Terms:", font=header_font, fill=(30,60,120))
    draw.text((40, y+22), "Please transfer payment within 30 days. Bank: HDFC | Acc: 123456789 | IFSC: HDFC0001234",
              font=small_font, fill=(80,80,80))

    # ── Footer ──
    draw.rectangle([0, H-50, W, H], fill=(30,60,120))
    draw.text((40, H-35), "Thank you for your business!", font=body_font, fill=(255,255,255))
    draw.text((W-280, H-35), f"www.{company.split()[0].lower()}.com", font=body_font, fill=(200,220,255))

    # Apply slight noise (simulate scan)
    img = add_scan_noise(img, level='light')
    path = os.path.join(SAMPLES_DIR, f"invoice_{idx:03d}.png")
    img.save(path)
    return path, inv_num, inv_date, due_date, total, company, client


# ─── CONTRACT GENERATOR ───────────────────────────────────────────────────────
def generate_contract(idx=1):
    """
    Creates an unstructured free-text Contract image with:
    - Title, parties, clauses
    - Signature block
    """
    W, H = 794, 1123
    img = Image.new('RGB', (W, H), color=(252, 252, 248))
    draw = ImageDraw.Draw(img)

    title_font  = get_bold_font(22)
    header_font = get_bold_font(13)
    body_font   = get_font(12)
    small_font  = get_font(10)

    party_a = random.choice(["Apex Solutions Ltd.", "NovaTech Corp.", "GlobalPrime Inc."])
    party_b = random.choice(["TechVentures Pvt Ltd", "Horizon Systems", "CloudBase Inc."])
    contract_num = f"CON-{random.randint(1000,9999)}-{random.randint(100,999)}"
    date = f"{random.randint(1,28):02d}/{random.randint(1,12):02d}/2024"
    value = random.randint(50000, 500000)

    # Header
    draw.text((W//2 - 180, 40), "SERVICE AGREEMENT CONTRACT", font=title_font, fill=(20,20,80))
    draw.line([60, 80, W-60, 80], fill=(20,20,80), width=2)
    draw.text((60, 95), f"Contract No: {contract_num}    Date: {date}", font=small_font, fill=(100,100,100))

    # Parties
    y = 130
    draw.text((60, y), "PARTIES:", font=header_font, fill=(20,20,80))
    y += 22
    clauses = [
        (f"This Service Agreement ('Agreement') is entered into on {date} between {party_a} "
         f"('Service Provider'), a company incorporated under the laws of India, with its "
         f"registered office at 123 Corporate Park, Pune, Maharashtra - 411001, AND"),
        (f"{party_b} ('Client'), a company incorporated under the laws of India, with its "
         f"registered office at 456 Client Avenue, Mumbai, Maharashtra - 400001."),
    ]
    for cl in clauses:
        for line in textwrap.wrap(cl, width=90):
            draw.text((60, y), line, font=body_font, fill=(40,40,40))
            y += 18
        y += 8

    # Clauses
    contract_clauses = [
        ("1. SCOPE OF SERVICES",
         f"The Service Provider agrees to provide software development, cloud infrastructure "
         f"management, and technical consulting services as detailed in Schedule A attached hereto. "
         f"All deliverables shall conform to the specifications agreed upon in writing by both parties."),
        ("2. CONTRACT VALUE & PAYMENT",
         f"The total contract value is INR {value:,}/- (Rupees {value:,} only) exclusive of applicable "
         f"taxes. Payment shall be made in equal monthly instalments. Late payment shall attract "
         f"interest at 18% per annum on the outstanding amount."),
        ("3. TERM & TERMINATION",
         f"This Agreement shall commence on {date} and continue for a period of 12 (twelve) months. "
         f"Either party may terminate this Agreement with 30 days written notice. Termination for "
         f"cause may be immediate upon material breach by the other party."),
        ("4. CONFIDENTIALITY",
         f"Both parties agree to maintain strict confidentiality of all proprietary information, "
         f"trade secrets, and business data shared during the term of this Agreement. This obligation "
         f"survives termination for a period of 3 years."),
        ("5. INTELLECTUAL PROPERTY",
         f"All work product, code, and deliverables created under this Agreement shall be the "
         f"exclusive property of {party_b} upon full payment. The Service Provider retains rights "
         f"to pre-existing tools and frameworks used in service delivery."),
        ("6. GOVERNING LAW",
         f"This Agreement shall be governed by the laws of India. Disputes shall be resolved by "
         f"arbitration in Pune, Maharashtra under the Arbitration and Conciliation Act, 1996."),
    ]

    for clause_title, clause_text in contract_clauses:
        if y > H - 200:
            break
        y += 10
        draw.text((60, y), clause_title, font=header_font, fill=(20,20,80))
        y += 20
        for line in textwrap.wrap(clause_text, width=90):
            if y > H - 180:
                break
            draw.text((60, y), line, font=body_font, fill=(40,40,40))
            y += 17
        y += 5

    # Signature block
    y = max(y + 30, H - 170)
    draw.line([60, y, W-60, y], fill=(150,150,150), width=1)
    y += 15
    draw.text((60,  y), "FOR SERVICE PROVIDER:",       font=header_font, fill=(20,20,80))
    draw.text((420, y), "FOR CLIENT:",                  font=header_font, fill=(20,20,80))
    y += 50
    draw.line([60,  y, 300, y], fill=(80,80,80), width=1)
    draw.line([420, y, 660, y], fill=(80,80,80), width=1)
    y += 8
    draw.text((60,  y), f"Authorised Signatory — {party_a}",  font=small_font, fill=(80,80,80))
    draw.text((420, y), f"Authorised Signatory — {party_b}",  font=small_font, fill=(80,80,80))
    y += 18
    draw.text((60,  y), f"Date: ___________",  font=small_font, fill=(100,100,100))
    draw.text((420, y), f"Date: ___________",  font=small_font, fill=(100,100,100))

    img = add_scan_noise(img, level='medium')
    path = os.path.join(SAMPLES_DIR, f"contract_{idx:03d}.png")
    img.save(path)
    return path, contract_num, date, value, party_a, party_b


# ─── REPORT GENERATOR ─────────────────────────────────────────────────────────
def generate_report(idx=1):
    """
    Creates a mixed structured/unstructured Financial Report image with:
    - Executive summary (free text)
    - Key metrics table (structured)
    - Analysis paragraphs
    """
    W, H = 794, 1123
    img = Image.new('RGB', (W, H), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)

    title_font  = get_bold_font(24)
    header_font = get_bold_font(13)
    body_font   = get_font(11)
    small_font  = get_font(10)

    company  = random.choice(["Apex Solutions Ltd.", "NovaTech Corp.", "GlobalPrime Inc."])
    quarter  = random.choice(["Q1 2024", "Q2 2024", "Q3 2024", "Q4 2023"])
    revenue  = random.randint(10, 50) * 100000
    profit   = int(revenue * random.uniform(0.08, 0.25))
    expenses = revenue - profit
    growth   = random.uniform(5, 35)

    # Header
    draw.rectangle([0, 0, W, 100], fill=(15, 40, 80))
    draw.text((40, 18), company, font=title_font, fill=(255,255,255))
    draw.text((40, 58), f"QUARTERLY FINANCIAL REPORT — {quarter}", font=header_font, fill=(180, 210, 255))
    draw.text((40, 82), "CONFIDENTIAL — FOR INTERNAL USE ONLY", font=small_font, fill=(120,160,220))

    # Report ID
    report_id = f"RPT-{random.randint(1000,9999)}"
    draw.text((W-160, 40), f"Report ID: {report_id}", font=small_font, fill=(180,210,255))
    draw.text((W-160, 58), f"Date: {random.randint(1,28):02d}/2024", font=small_font, fill=(180,210,255))

    y = 120
    # Executive Summary
    draw.text((40, y), "EXECUTIVE SUMMARY", font=header_font, fill=(15,40,80))
    y += 22
    summary = (
        f"{company} achieved strong performance during {quarter}, recording total revenue of "
        f"INR {revenue:,}/- against operating expenses of INR {expenses:,}/-. "
        f"Net profit stood at INR {profit:,}/-, representing a margin of {profit/revenue*100:.1f}%. "
        f"Year-over-year growth of {growth:.1f}% reflects our continued expansion in cloud services "
        f"and enterprise software segments. The management team remains focused on operational "
        f"efficiency, talent acquisition, and strategic market penetration for the upcoming quarters."
    )
    for line in textwrap.wrap(summary, width=90):
        draw.text((40, y), line, font=body_font, fill=(40,40,40))
        y += 17
    y += 20

    # Key Metrics Table
    draw.text((40, y), "KEY FINANCIAL METRICS", font=header_font, fill=(15,40,80))
    y += 18
    draw.rectangle([40, y, W-40, y+30], fill=(15,40,80))
    draw.text((50,  y+8), "Metric",          font=header_font, fill=(255,255,255))
    draw.text((280, y+8), "Current Quarter", font=header_font, fill=(255,255,255))
    draw.text((500, y+8), "Previous Quarter",font=header_font, fill=(255,255,255))
    draw.text((660, y+8), "YoY Growth",      font=header_font, fill=(255,255,255))
    y += 30

    prev_revenue = int(revenue / (1 + growth/100))
    metrics = [
        ("Total Revenue",   f"₹{revenue:,}",          f"₹{prev_revenue:,}",      f"+{growth:.1f}%"),
        ("Net Profit",      f"₹{profit:,}",           f"₹{int(profit*0.85):,}",  f"+{random.uniform(3,20):.1f}%"),
        ("Operating Exp.",  f"₹{expenses:,}",         f"₹{int(expenses*1.05):,}",f"-{random.uniform(1,8):.1f}%"),
        ("EBITDA Margin",   f"{profit/revenue*100:.1f}%", f"{profit/revenue*100*0.9:.1f}%", f"+{random.uniform(1,5):.1f}%"),
        ("Client Count",    str(random.randint(45,200)),   str(random.randint(35,150)),  f"+{random.randint(5,30)}%"),
        ("Emp. Count",      str(random.randint(100,500)),  str(random.randint(80,400)),  f"+{random.randint(2,15)}%"),
    ]
    for i, (metric, cur, prev, growth_val) in enumerate(metrics):
        bg = (235, 243, 255) if i % 2 == 0 else (248, 250, 255)
        draw.rectangle([40, y, W-40, y+25], fill=bg)
        draw.text((50,  y+5), metric,     font=body_font, fill=(30,30,30))
        draw.text((280, y+5), cur,        font=body_font, fill=(15,100,15) if "+" in (cur or "") else (30,30,30))
        draw.text((500, y+5), prev,       font=body_font, fill=(80,80,80))
        color = (0, 130, 0) if "+" in growth_val else (180, 0, 0)
        draw.text((660, y+5), growth_val, font=body_font, fill=color)
        y += 25
    y += 20

    # Analysis Section
    sections = [
        ("REVENUE ANALYSIS",
         f"Revenue growth of {growth:.1f}% was primarily driven by expansion in the enterprise "
         f"segment, which contributed 65% of total revenue. New client acquisitions in the "
         f"manufacturing and BFSI verticals added significantly to the top line."),
        ("RISK FACTORS",
         f"Key risks include currency fluctuation, increasing competition in the cloud services "
         f"market, and potential regulatory changes. The management has implemented hedging "
         f"strategies and diversified the client portfolio to mitigate these risks."),
        ("OUTLOOK",
         f"The company maintains a positive outlook for the next quarter with a projected revenue "
         f"growth of {growth*1.1:.1f}%. Three new product launches are planned, targeting the "
         f"healthcare and education sectors, which are expected to contribute to revenue growth."),
    ]
    for sec_title, sec_text in sections:
        if y > H - 80:
            break
        draw.text((40, y), sec_title, font=header_font, fill=(15,40,80))
        y += 20
        for line in textwrap.wrap(sec_text, width=90):
            if y > H - 60:
                break
            draw.text((40, y), line, font=body_font, fill=(40,40,40))
            y += 16
        y += 12

    # Footer
    draw.rectangle([0, H-45, W, H], fill=(15,40,80))
    draw.text((40, H-32), f"© 2024 {company} | Confidential", font=small_font, fill=(180,210,255))
    draw.text((W-200, H-32), f"Page 1 of 1 | {report_id}",    font=small_font, fill=(180,210,255))

    img = add_scan_noise(img, level='light')
    path = os.path.join(SAMPLES_DIR, f"report_{idx:03d}.png")
    img.save(path)
    return path, report_id, quarter, revenue, profit, company


# ─── SCAN NOISE SIMULATOR ─────────────────────────────────────────────────────
def add_scan_noise(img, level='light'):
    """
    Simulates real scanned document artifacts:
    - Gaussian noise
    - Slight rotation (skew)
    - Brightness variation
    Level: 'light', 'medium', 'heavy'
    """
    arr = np.array(img, dtype=np.float32)

    # Gaussian noise
    noise_std = {'light': 3, 'medium': 8, 'heavy': 18}[level]
    noise = np.random.normal(0, noise_std, arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    img = Image.fromarray(arr)

    # Slight rotation to simulate skew
    angle = {'light': 0.3, 'medium': 0.8, 'heavy': 2.0}[level]
    angle = random.uniform(-angle, angle)
    img = img.rotate(angle, fillcolor=(255, 255, 255))

    # Slight blur (scanner softness)
    if level in ('medium', 'heavy'):
        img = img.filter(ImageFilter.GaussianBlur(radius=0.5))

    return img


# ─── GENERATE ALL SAMPLES ─────────────────────────────────────────────────────
def generate_all_samples(n_each=5):
    """Generate n_each of each document type for the dataset."""
    generated = []
    print("📄 Generating sample documents...")
    for i in range(1, n_each+1):
        path, *meta = generate_invoice(i)
        generated.append({'type': 'invoice', 'path': path, 'meta': meta})
        print(f"  ✅ Invoice {i}: {os.path.basename(path)}")

    for i in range(1, n_each+1):
        path, *meta = generate_contract(i)
        generated.append({'type': 'contract', 'path': path, 'meta': meta})
        print(f"  ✅ Contract {i}: {os.path.basename(path)}")

    for i in range(1, n_each+1):
        path, *meta = generate_report(i)
        generated.append({'type': 'report', 'path': path, 'meta': meta})
        print(f"  ✅ Report {i}: {os.path.basename(path)}")

    print(f"\n✔ Generated {len(generated)} sample documents in: {SAMPLES_DIR}")
    return generated


if __name__ == "__main__":
    generate_all_samples(n_each=5)

In [ ]:
"""
=============================================================
STEP 2: IMAGE PREPROCESSING PIPELINE
=============================================================
Cleans and enhances scanned document images before OCR:
  1. Grayscale conversion
  2. Noise removal (Gaussian + Median filters)
  3. Skew/deskew correction
  4. Brightness & contrast normalization (CLAHE)
  5. Thresholding (Otsu's binarization)
  6. Morphological cleanup (dilation/erosion to fix broken chars)
  7. Resize to optimal OCR resolution (300 DPI equivalent)
"""

import cv2
import numpy as np
from PIL import Image
import os

# ─── PREPROCESSING CLASS ──────────────────────────────────────────────────────

class DocumentPreprocessor:
    """
    Full preprocessing pipeline for scanned corporate documents.
    Each method can be used independently or chained via `process()`.
    """

    def __init__(self, target_dpi=300, target_width=2480):
        """
        Args:
            target_dpi    : Target DPI for OCR (300 is optimal for Tesseract)
            target_width  : Target width in pixels (A4 @ 300dpi ≈ 2480px)
        """
        self.target_dpi   = target_dpi
        self.target_width = target_width

    # ── STEP 2a: Load image (PIL or path) ─────────────────────────────────────
    def load(self, source):
        """Load from file path, PIL Image, or numpy array."""
        if isinstance(source, str):
            img = cv2.imread(source)
            if img is None:
                raise FileNotFoundError(f"Cannot load image: {source}")
        elif isinstance(source, Image.Image):
            img = cv2.cvtColor(np.array(source), cv2.COLOR_RGB2BGR)
        elif isinstance(source, np.ndarray):
            img = source.copy()
        else:
            raise TypeError("Source must be file path, PIL Image, or numpy array")
        return img

    # ── STEP 2b: Grayscale conversion ─────────────────────────────────────────
    def to_grayscale(self, img):
        """Convert BGR image to grayscale for processing."""
        if len(img.shape) == 3:
            return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return img  # already grayscale

    # ── STEP 2c: Resize to target width (maintain aspect ratio) ───────────────
    def resize(self, img):
        """
        Scale image to target_width while preserving aspect ratio.
        Larger images = better OCR accuracy.
        """
        h, w = img.shape[:2]
        if w == self.target_width:
            return img
        scale = self.target_width / w
        new_h = int(h * scale)
        # Use LANCZOS for downscaling, CUBIC for upscaling
        interp = cv2.INTER_LANCZOS4 if scale < 1 else cv2.INTER_CUBIC
        return cv2.resize(img, (self.target_width, new_h), interpolation=interp)

    # ── STEP 2d: Noise removal ─────────────────────────────────────────────────
    def remove_noise(self, gray):
        """
        Two-stage noise removal:
        1. Gaussian blur  — smooths random pixel noise
        2. Median filter  — removes salt-and-pepper noise (scanner dust)
        """
        # Gaussian: kernel 3x3, sigma=0 (auto)
        blurred = cv2.GaussianBlur(gray, (3, 3), 0)
        # Median: kernel 3 (odd number)
        denoised = cv2.medianBlur(blurred, 3)
        return denoised

    # ── STEP 2e: Brightness & contrast normalization (CLAHE) ──────────────────
    def normalize_brightness(self, gray):
        """
        CLAHE (Contrast Limited Adaptive Histogram Equalization):
        - Enhances local contrast adaptively
        - Helps with uneven lighting from scanner
        - clipLimit: max contrast amplification
        - tileGridSize: region size for local equalization
        """
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return clahe.apply(gray)

    # ── STEP 2f: Deskew (correct rotation/skew) ───────────────────────────────
    def deskew(self, gray):
        """
        Detects and corrects document skew using Hough line transform.
        Scanned documents are often slightly rotated (±2°).
        """
        try:
            # Invert (white text on black for edge detection)
            inverted = cv2.bitwise_not(gray)
            # Threshold to binary
            _, thresh = cv2.threshold(inverted, 10, 255, cv2.THRESH_BINARY)
            # Find coordinates of all non-zero pixels
            coords = np.column_stack(np.where(thresh > 0))
            if len(coords) < 100:
                return gray  # Not enough content to detect angle
            # Minimum area rectangle gives us the angle
            angle = cv2.minAreaRect(coords)[-1]
            # minAreaRect returns angles in range (-90, 0]
            if angle < -45:
                angle = 90 + angle
            elif angle > 45:
                angle = angle - 90
            # Only correct if skew is significant (> 0.2°) but not extreme
            if abs(angle) < 0.2 or abs(angle) > 10:
                return gray
            # Rotate image
            h, w = gray.shape[:2]
            center = (w // 2, h // 2)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            rotated = cv2.warpAffine(gray, M, (w, h),
                                     flags=cv2.INTER_CUBIC,
                                     borderMode=cv2.BORDER_REPLICATE)
            return rotated
        except Exception:
            return gray  # Return original if deskew fails

    # ── STEP 2g: Binarization (Otsu's thresholding) ───────────────────────────
    def binarize(self, gray):
        """
        Otsu's method automatically finds the optimal threshold:
        - Converts grayscale to pure black/white
        - Maximizes inter-class variance between text and background
        - Essential for clean OCR input
        """
        # Otsu's global threshold
        _, binary = cv2.threshold(gray, 0, 255,
                                   cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binary

    # ── STEP 2h: Adaptive thresholding (alternative for poor lighting) ─────────
    def adaptive_binarize(self, gray):
        """
        Adaptive thresholding — better for documents with shadows or gradients.
        Computes threshold per local region instead of globally.
        """
        return cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            blockSize=11,  # local region size
            C=2             # constant subtracted from mean
        )

    # ── STEP 2i: Morphological cleanup ────────────────────────────────────────
    def morphological_cleanup(self, binary):
        """
        Morphological operations to fix OCR artifacts:
        - Erosion  : removes small noise blobs
        - Dilation : reconnects broken character strokes
        - Opening  : erosion then dilation (removes noise)
        """
        kernel = np.ones((1, 1), np.uint8)
        # Remove small noise pixels
        cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
        # Connect broken strokes in characters
        kernel2 = np.ones((1, 1), np.uint8)
        cleaned = cv2.dilate(cleaned, kernel2, iterations=1)
        return cleaned

    # ── STEP 2j: Remove borders/artifacts (page borders from scanner) ──────────
    def remove_borders(self, binary):
        """
        Remove dark scanner borders that appear around scanned pages.
        Uses contour detection to find and mask border regions.
        """
        # Find contours
        contours, _ = cv2.findContours(
            cv2.bitwise_not(binary), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        h, w = binary.shape
        mask = np.ones_like(binary) * 255  # white mask

        for cnt in contours:
            x, y, cw, ch = cv2.boundingRect(cnt)
            # Skip contours touching or very near the image border
            border_margin = 5
            if (x <= border_margin or y <= border_margin or
                    x + cw >= w - border_margin or y + ch >= h - border_margin):
                # This is a border artifact — fill it white
                cv2.rectangle(mask, (x, y), (x + cw, y + ch), 255, -1)

        return cv2.bitwise_and(binary, mask)

    # ── MAIN PIPELINE ─────────────────────────────────────────────────────────
    def process(self, source, save_path=None, use_adaptive=False):
        """
        Full preprocessing pipeline:
        load → grayscale → resize → denoise → normalize →
        deskew → binarize → morphological cleanup → (optional save)

        Args:
            source       : image path, PIL Image, or numpy array
            save_path    : if provided, saves preprocessed image here
            use_adaptive : use adaptive thresholding (better for shadows)

        Returns:
            preprocessed numpy array (grayscale, binary)
        """
        img  = self.load(source)
        gray = self.to_grayscale(img)
        gray = self.resize(gray)
        gray = self.remove_noise(gray)
        gray = self.normalize_brightness(gray)
        gray = self.deskew(gray)

        if use_adaptive:
            binary = self.adaptive_binarize(gray)
        else:
            binary = self.binarize(gray)

        binary = self.morphological_cleanup(binary)

        if save_path:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            cv2.imwrite(save_path, binary)

        return binary

    # ── QUICK PREVIEW: side-by-side before/after ──────────────────────────────
    def get_comparison(self, source):
        """Returns (original_gray, preprocessed) tuple for display."""
        img  = self.load(source)
        gray = self.to_grayscale(img)
        # Resize original for display
        h, w = gray.shape[:2]
        display_w = min(w, 800)
        scale = display_w / w
        display = cv2.resize(gray, (display_w, int(h * scale)))
        processed = self.process(source)
        proc_display = cv2.resize(processed, (display_w, int(processed.shape[0] * scale)))
        return display, proc_display


# ─── Convenience function ─────────────────────────────────────────────────────
def preprocess_image(image_source, save_path=None, use_adaptive=False):
    """
    Convenience wrapper around DocumentPreprocessor.process().
    Accepts file path, PIL Image, or numpy array.
    Returns preprocessed binary image as numpy array.
    """
    preprocessor = DocumentPreprocessor()
    return preprocessor.process(image_source, save_path=save_path, use_adaptive=use_adaptive)


if __name__ == "__main__":
    # Test preprocessing on a sample image
    import sys
    test_path = sys.argv[1] if len(sys.argv) > 1 else None

    if test_path and os.path.exists(test_path):
        preprocessor = DocumentPreprocessor()
        result = preprocessor.process(test_path, save_path="/tmp/preprocessed_test.png")
        print(f"✅ Preprocessed: {test_path}")
        print(f"   Output shape: {result.shape}")
        print(f"   Saved to: /tmp/preprocessed_test.png")
    else:
        print("Usage: python preprocess.py <image_path>")
        print("No test image provided — preprocessor module loaded OK.")

In [ ]:
"""
=============================================================
STEP 3A: OCR IMPLEMENTATION — Tesseract + OpenCV Engine
=============================================================
Extracts text from preprocessed document images using:
  - Tesseract OCR (primary engine)
  - OpenCV preprocessing integration
  - Text normalization and error correction
  - Tokenization for downstream NLP
"""

import re
import os
import pytesseract
import cv2
import numpy as np
from PIL import Image
from utils.preprocess import DocumentPreprocessor

# ─── OCR ENGINE CLASS ─────────────────────────────────────────────────────────

class OCREngine:
    """
    Tesseract-based OCR engine with:
    - Multiple PSM (Page Segmentation Mode) strategies
    - Confidence scoring
    - Text normalization pipeline
    - Structured data extraction helpers
    """

    # Tesseract Page Segmentation Modes (PSM)
    PSM_AUTO        = 3   # Fully automatic — best for most documents
    PSM_SINGLE_BLOCK= 6   # Assume a single uniform block of text
    PSM_SINGLE_LINE = 7   # Single text line
    PSM_SPARSE_TEXT = 11  # Find as much text as possible, no order assumed

    def __init__(self):
        self.preprocessor = DocumentPreprocessor()
        # Verify Tesseract is installed
        try:
            version = pytesseract.get_tesseract_version()
            print(f"✅ Tesseract version: {version}")
        except Exception as e:
            print(f"⚠️  Tesseract not found: {e}")

    # ── CORE OCR METHOD ────────────────────────────────────────────────────────
    def extract_text(self, image_source, psm=3, lang='eng', preprocess=True):
        """
        Extract raw text from an image using Tesseract.

        Args:
            image_source : path, PIL Image, or numpy array
            psm          : Tesseract Page Segmentation Mode
            lang         : OCR language (default 'eng')
            preprocess   : apply preprocessing pipeline before OCR

        Returns:
            dict with keys: text, confidence, word_count
        """
        # Load image
        if isinstance(image_source, str):
            img = cv2.imread(image_source)
            if img is None:
                return {'text': '', 'confidence': 0, 'word_count': 0}
        elif isinstance(image_source, Image.Image):
            img = cv2.cvtColor(np.array(image_source), cv2.COLOR_RGB2BGR)
        else:
            img = image_source.copy()

        # Preprocess for better OCR accuracy
        if preprocess:
            # Try Otsu first, then adaptive if result is poor
            binary = self.preprocessor.process(img)
            pil_img = Image.fromarray(binary)
        else:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
            pil_img = Image.fromarray(gray)

        # Tesseract config
        config = f'--psm {psm} --oem 3'  # OEM 3 = LSTM + legacy engine

        try:
            # Get text with confidence data
            data = pytesseract.image_to_data(
                pil_img, config=config, lang=lang,
                output_type=pytesseract.Output.DICT
            )

            # Filter confident words (confidence > 0)
            words      = []
            confidences= []
            for i, word in enumerate(data['text']):
                if word.strip() and data['conf'][i] > 0:
                    words.append(word)
                    confidences.append(data['conf'][i])

            # Full text extraction
            raw_text = pytesseract.image_to_string(pil_img, config=config, lang=lang)
            avg_conf = float(np.mean(confidences)) if confidences else 0.0

            return {
                'text'      : raw_text,
                'confidence': round(avg_conf, 2),
                'word_count': len(words),
                'words'     : words,
                'word_confidences': confidences
            }

        except Exception as e:
            return {'text': '', 'confidence': 0, 'word_count': 0, 'error': str(e)}

    # ── MULTI-STRATEGY EXTRACTION ─────────────────────────────────────────────
    def extract_best(self, image_source):
        """
        Try multiple PSM modes and return the result with highest confidence.
        Good for documents where layout is unknown.
        """
        results = []
        for psm in [3, 6, 11]:
            r = self.extract_text(image_source, psm=psm)
            results.append(r)

        # Return the result with the most words AND decent confidence
        best = max(results, key=lambda r: r['word_count'] * (r['confidence'] / 100 + 0.01))
        return best

    # ── TEXT NORMALIZATION ─────────────────────────────────────────────────────
    def normalize_text(self, raw_text):
        """
        Post-OCR text normalization:
        1. Fix common OCR character confusions (0→O, l→1, etc.)
        2. Normalize whitespace
        3. Fix broken hyphenation
        4. Normalize currency and number formats
        5. Remove control characters

        Returns cleaned text string.
        """
        text = raw_text

        # ① Remove non-printable control characters (except newlines/tabs)
        text = re.sub(r'[^\x09\x0A\x0D\x20-\x7E\u00A0-\uFFFF]', '', text)

        # ② Fix broken lines (hyphenated word continuation)
        text = re.sub(r'-\n(\w)', r'\1', text)

        # ③ Collapse multiple spaces/tabs to single space
        text = re.sub(r'[ \t]+', ' ', text)

        # ④ Collapse 3+ newlines to 2 (preserve paragraph breaks)
        text = re.sub(r'\n{3,}', '\n\n', text)

        # ⑤ Fix common OCR confusions in numbers
        # In a numeric context, 'O' → '0', 'l'/'I' → '1'
        text = re.sub(r'(?<=\d)O(?=\d)', '0', text)   # 3O5 → 305
        text = re.sub(r'(?<=\d)l(?=\d)', '1', text)   # 3l5 → 315
        text = re.sub(r'(?<=\d)I(?=\d)', '1', text)   # 3I5 → 315

        # ⑥ Fix currency symbols (OCR often gets $ wrong as S or ¥)
        text = re.sub(r'\bINR\b|\bRs\.?\b', '₹', text)
        text = re.sub(r'(?<!\w)\$\s*(?=\d)', '$', text)

        # ⑦ Normalize date patterns (ensure consistent spacing)
        text = re.sub(r'(\d{1,2})\s*/\s*(\d{1,2})\s*/\s*(\d{2,4})', r'\1/\2/\3', text)

        # ⑧ Fix invoice/contract number patterns (remove extra spaces inside)
        text = re.sub(r'(INV|CON|RPT)\s*-\s*(\w+)', r'\1-\2', text)

        # ⑨ Strip leading/trailing whitespace
        text = text.strip()

        return text

    # ── TOKENIZATION ──────────────────────────────────────────────────────────
    def tokenize(self, text):
        """
        Simple rule-based tokenizer (no NLTK dependency):
        - Sentence tokenization via regex
        - Word tokenization
        - Returns structured tokens dict
        """
        # Sentence split: split on . ! ? followed by space+Capital
        sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
        sentences = [s.strip() for s in sentences if s.strip()]

        # Word tokenization: split on non-word chars, keep alphanumeric + some punctuation
        words = re.findall(r"\b[\w'$₹.,:-]+\b", text)

        # Filter very short tokens (likely noise)
        meaningful_words = [w for w in words if len(w) >= 2]

        return {
            'sentences'       : sentences,
            'words'           : meaningful_words,
            'sentence_count'  : len(sentences),
            'word_count'      : len(meaningful_words),
            'unique_words'    : len(set(w.lower() for w in meaningful_words))
        }

    # ── FULL PIPELINE ─────────────────────────────────────────────────────────
    def process_document(self, image_source):
        """
        Complete OCR pipeline:
        extract → normalize → tokenize

        Returns:
            dict with all OCR results and processed text
        """
        # OCR extraction
        ocr_result = self.extract_best(image_source)
        raw_text   = ocr_result.get('text', '')

        # Normalize text
        clean_text = self.normalize_text(raw_text)

        # Tokenize
        tokens = self.tokenize(clean_text)

        return {
            'raw_text'   : raw_text,
            'clean_text' : clean_text,
            'confidence' : ocr_result['confidence'],
            'word_count' : ocr_result['word_count'],
            'tokens'     : tokens,
        }


# ─── Run standalone test ──────────────────────────────────────────────────────
if __name__ == "__main__":
    import sys
    path = sys.argv[1] if len(sys.argv) > 1 else None
    if path and os.path.exists(path):
        engine = OCREngine()
        result = engine.process_document(path)
        print(f"\n{'='*60}")
        print(f"OCR RESULT")
        print(f"{'='*60}")
        print(f"Confidence   : {result['confidence']:.1f}%")
        print(f"Word Count   : {result['word_count']}")
        print(f"Sentences    : {result['tokens']['sentence_count']}")
        print(f"\n--- Cleaned Text (first 500 chars) ---")
        print(result['clean_text'][:500])
    else:
        print("Usage: python ocr_engine.py <image_path>")

In [ ]:
"""
=============================================================
STEP 3B: NLP FIELD EXTRACTION + ML DOCUMENT CLASSIFICATION
=============================================================
Two components:
  A) FieldExtractor — rule-based NLP to extract structured fields
     from OCR text (dates, amounts, IDs, names, parties)
  B) DocumentClassifier — ML model (TF-IDF + SVM) trained to
     classify documents into: invoice / contract / report
"""

import re
import os
import pickle
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ─────────────────────────────────────────────────────────────────────────────
# PART A: FIELD EXTRACTOR (Rule-based NLP)
# ─────────────────────────────────────────────────────────────────────────────

class FieldExtractor:
    """
    Extracts key business entities from OCR text using regex patterns.
    Categories extracted:
      - Dates          (DD/MM/YYYY, Month DD YYYY, etc.)
      - Amounts/Money  (₹, $, INR, numbers with commas)
      - Document IDs   (INV-xxxxx, CON-xxxx, RPT-xxxx)
      - Organization   (Ltd, Corp, Inc, Pvt patterns)
      - GST Numbers    (Indian GST format)
      - Email/Phone    (contact info)
      - Percentages    (tax rates, growth %)
    """

    def __init__(self):
        # ── Compiled regex patterns (compiled once for performance) ─────────
        self.patterns = {

            # Dates: DD/MM/YYYY or YYYY-MM-DD or Month DD, YYYY
            'dates': [
                re.compile(r'\b(\d{1,2})[/\-\.](\d{1,2})[/\-\.](\d{2,4})\b'),
                re.compile(r'\b(January|February|March|April|May|June|July|'
                           r'August|September|October|November|December)\s+(\d{1,2}),?\s+(\d{4})\b',
                           re.IGNORECASE),
                re.compile(r'\b(\d{4})[/\-](\d{1,2})[/\-](\d{1,2})\b'),
                re.compile(r'\b(Q[1-4]\s+\d{4})\b'),     # Quarter format: Q3 2024
            ],

            # Money amounts: $1,234.56 or ₹1,23,456 or INR 50,000
            'amounts': [
                re.compile(r'(?:₹|\$|INR|Rs\.?)\s*(\d{1,3}(?:[,\s]\d{2,3})*(?:\.\d{2})?)\b'),
                re.compile(r'\b(\d{1,3}(?:,\d{2,3})+(?:\.\d{2})?)\s*(?:/-|INR|₹|\$)'),
                re.compile(r'\bINR\s+(\d[\d,]+(?:\.\d{2})?)\b'),
                re.compile(r'\b(\d+(?:\.\d{2})?)\s*(?:lakhs?|crores?)\b', re.IGNORECASE),
            ],

            # Invoice / Contract / Report IDs
            'document_ids': [
                re.compile(r'\b(INV[-/]\d{3,8})\b', re.IGNORECASE),
                re.compile(r'\b(CON[-/]\d{3,8}[-/]?\d*)\b', re.IGNORECASE),
                re.compile(r'\b(RPT[-/]\d{3,8})\b', re.IGNORECASE),
                re.compile(r'\b(Contract\s+No[:.]\s*[\w\-]+)\b', re.IGNORECASE),
                re.compile(r'\b(Invoice\s+#[:\s]*[\w\-]+)\b', re.IGNORECASE),
                re.compile(r'\b(Report\s+ID[:\s]*[\w\-]+)\b', re.IGNORECASE),
            ],

            # Organizations (company names)
            'organizations': [
                re.compile(r'\b([A-Z][A-Za-z\s]+(?:Ltd\.?|Limited|Corp\.?|Corporation|'
                           r'Inc\.?|Incorporated|Pvt\.?\s*Ltd\.?|LLP|LLC|Co\.))\b'),
            ],

            # Indian GST number: 2-digit state + 10-digit PAN + 1 + Z + 1
            'gst_numbers': [
                re.compile(r'\b(\d{2}[A-Z]{5}\d{4}[A-Z]{1}[A-Z\d]{1}Z[A-Z\d]{1})\b'),
            ],

            # Email addresses
            'emails': [
                re.compile(r'\b([a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,})\b'),
            ],

            # Phone numbers (Indian: 10 digits, or with +91)
            'phones': [
                re.compile(r'\b(?:\+91[\s\-]?)?([6-9]\d{9})\b'),
                re.compile(r'\b(\d{3}[\s\-]\d{3}[\s\-]\d{4})\b'),
            ],

            # Percentages
            'percentages': [
                re.compile(r'\b(\d+(?:\.\d+)?)\s*%'),
            ],

            # Payment terms keywords
            'payment_terms': [
                re.compile(r'\b(Net\s+\d+|Due\s+on\s+receipt|Immediate|'
                           r'30\s+days?|60\s+days?|90\s+days?)\b', re.IGNORECASE),
            ],

            # Tax rates (GST 18%, TDS, etc.)
            'tax_info': [
                re.compile(r'\b(GST|CGST|SGST|IGST|TDS|VAT|Tax)\s*(?:@|at|:)?\s*'
                           r'(\d+(?:\.\d+)?)\s*%', re.IGNORECASE),
            ],
        }

    def extract(self, text):
        """
        Run all pattern extractors on text.
        Returns dict of all extracted fields.
        """
        results = {}

        for field_name, patterns in self.patterns.items():
            matches = []
            for pattern in patterns:
                found = pattern.findall(text)
                for match in found:
                    # Flatten tuple matches to string
                    if isinstance(match, tuple):
                        val = ' '.join(str(m) for m in match if m).strip()
                    else:
                        val = str(match).strip()
                    if val and val not in matches:
                        matches.append(val)
            results[field_name] = matches

        # ── Derived fields ────────────────────────────────────────────────────
        # Primary document ID (first found)
        results['primary_id'] = results['document_ids'][0] if results['document_ids'] else None

        # Primary date (first found)
        results['primary_date'] = results['dates'][0] if results['dates'] else None

        # Total amount (largest amount found)
        if results['amounts']:
            try:
                parsed = []
                for amt in results['amounts']:
                    num = re.sub(r'[^\d.]', '', str(amt))
                    if num:
                        parsed.append(float(num))
                results['max_amount'] = max(parsed) if parsed else None
            except:
                results['max_amount'] = None
        else:
            results['max_amount'] = None

        # Summary string for display
        results['summary'] = self._build_summary(results)
        return results

    def _build_summary(self, fields):
        """Build a human-readable summary of extracted fields."""
        parts = []
        if fields.get('primary_id'):
            parts.append(f"ID: {fields['primary_id']}")
        if fields.get('primary_date'):
            parts.append(f"Date: {fields['primary_date']}")
        if fields.get('max_amount'):
            parts.append(f"Amount: ₹{fields['max_amount']:,.2f}")
        if fields.get('organizations'):
            parts.append(f"Org: {fields['organizations'][0]}")
        return ' | '.join(parts) if parts else 'No key fields extracted'


# ─────────────────────────────────────────────────────────────────────────────
# PART B: DOCUMENT CLASSIFIER (ML: TF-IDF + SVM)
# ─────────────────────────────────────────────────────────────────────────────

class DocumentClassifier:
    """
    ML-based document type classifier using:
    - TF-IDF vectorization (bag-of-words with term frequency weighting)
    - Support Vector Machine (SVM) with RBF kernel
    - Trained on keyword-rich synthetic text samples

    Classes: invoice | contract | report
    """

    CLASSES = ['invoice', 'contract', 'report']

    # Training corpus — representative text snippets for each class
    # In production, these would be real extracted OCR texts from labeled docs
    TRAINING_DATA = {
        'invoice': [
            "Invoice number INV-12345 date 15/06/2024 due date 15/07/2024 bill to client "
            "subtotal GST 18% total amount payable payment terms 30 days bank transfer NEFT "
            "unit price quantity line items description services rendered professional fees",
            "Tax invoice GST invoice number billing address GSTIN HSN SAC code "
            "total invoice value IGST CGST SGST taxable value rate percent amount "
            "goods services supply place of supply reverse charge e-invoice IRN",
            "Invoice # services cloud hosting technical support software development "
            "amount due payment due date overdue penalty interest rate net 30 "
            "purchase order PO number vendor supplier customer billing",
            "Proforma invoice quotation estimate USD EUR INR rupees dollars "
            "discount applied total before tax after tax final amount payable "
            "advance payment milestone billing schedule installment",
            "Credit note debit note adjustment invoice original invoice reference "
            "credit amount reason for credit returned goods defective service "
            "refund processing payment received outstanding balance",
            "Tax invoice GSTIN 27ABCDE1234F1Z5 HSN code 998314 services rendered "
            "professional consulting advisory fee taxable supply exempted supply "
            "place of supply Maharashtra reverse charge mechanism applicable",
            "Monthly subscription invoice recurring billing annual plan "
            "pro-rated charges activation fee setup charges waived off "
            "auto-renewal terms cancellation policy refund policy",
            "Purchase order invoice matching three-way match vendor invoice "
            "approved PO goods receipt note GRN payment processing accounts payable "
            "invoice approval workflow ERP SAP system integration",
        ],
        'contract': [
            "Service agreement contract between parties hereby agree terms conditions "
            "scope of work deliverables milestones payment schedule intellectual property "
            "confidentiality non-disclosure governing law arbitration dispute resolution "
            "termination clause notice period breach remedy indemnification",
            "This agreement entered into on date between party A service provider "
            "and party B client whereas the parties wish to formalize their arrangement "
            "now therefore in consideration of the mutual covenants herein contained "
            "the parties agree as follows authorised signatory witness",
            "Non-disclosure agreement NDA confidential information proprietary data "
            "trade secrets business information disclosed receiving party shall maintain "
            "strict confidentiality not disclose to third parties permitted disclosure "
            "return destroy confidential information upon termination survival clause",
            "Employment contract offer letter joining date designation role "
            "compensation salary CTC gross net TDS deduction benefits perquisites "
            "probation period notice period non-compete clause non-solicitation "
            "code of conduct company policies termination misconduct",
            "Lease agreement landlord tenant property address rent amount security deposit "
            "maintenance charges lock-in period renewal option sub-letting prohibited "
            "alterations permission stamp duty registration charges eviction notice",
            "Master service agreement MSA statement of work SOW change order "
            "project scope timeline deliverable acceptance criteria SLA "
            "uptime availability penalty service credit escalation matrix "
            "liability limitation indemnification warranty disclaimer",
            "Software license agreement end user license EULA perpetual annual "
            "subscription named users concurrent users restriction on use "
            "reverse engineering decompilation distribution prohibited "
            "support maintenance upgrade patch version",
            "Shareholders agreement share purchase agreement SPA due diligence "
            "representations warranties conditions precedent completion obligations "
            "consideration escrow earn-out vesting schedule drag-along tag-along "
            "right of first refusal pre-emptive rights anti-dilution",
        ],
        'report': [
            "Quarterly financial report executive summary revenue profit margin "
            "EBITDA operating expenses year over year growth Q1 Q2 Q3 Q4 "
            "key performance indicators KPI metrics analysis outlook forecast "
            "balance sheet profit loss cash flow statement",
            "Annual report board of directors management discussion analysis "
            "corporate governance auditor report financial statements notes "
            "significant accounting policies depreciation capital expenditure "
            "working capital current ratio debt equity earnings per share",
            "Market research report industry analysis competitor landscape "
            "SWOT analysis opportunities threats growth drivers demand supply "
            "market size TAM SAM SOM customer segments pricing strategy "
            "recommendations conclusion methodology data sources",
            "Audit report internal audit external audit findings observations "
            "recommendations management response compliance risk assessment "
            "control environment significant deficiencies material weakness "
            "audit opinion qualified unqualified disclaimer adverse",
            "Project status report milestone completion percentage budget "
            "actual vs planned variance risk issues action items responsible "
            "person due date priority escalation resolved pending next steps "
            "dependencies blockers resource utilization burn rate",
            "Sales performance report regional breakdown product wise "
            "channel wise customer acquisition retention churn rate "
            "pipeline conversion funnel stage deal size average "
            "quota attainment target achievement incentive commission",
            "Risk assessment report risk register probability impact "
            "inherent risk residual risk control effectiveness "
            "risk appetite tolerance threshold mitigation strategy "
            "contingency plan business continuity disaster recovery",
            "ESG report environmental social governance sustainability "
            "carbon footprint emissions diversity inclusion CSR initiatives "
            "stakeholder engagement materiality assessment GRI framework "
            "SDG alignment reporting standards BRSR",
        ]
    }

    def __init__(self, model_path=None):
        self.model_path = model_path or os.path.join(
            os.path.dirname(__file__), '..', 'models', 'classifier.pkl'
        )
        self.pipeline = None
        self.label_encoder = LabelEncoder()
        self.label_encoder.fit(self.CLASSES)
        self._train()  # Auto-train on init

    def _build_training_set(self):
        """Build X (texts) and y (labels) from training data."""
        X, y = [], []
        for label, texts in self.TRAINING_DATA.items():
            for text in texts:
                # Augment each sample with variations
                X.append(text)
                y.append(label)
                # Add a slightly modified version (simulates variation)
                X.append(text.replace(' ', '  ').lower())
                y.append(label)
        return X, y

    def _train(self):
        """
        Train the TF-IDF + SVM pipeline.
        TF-IDF: converts text to weighted term-frequency matrix
        SVM: finds optimal hyperplane to separate document classes
        """
        X, y = self._build_training_set()

        # Build sklearn Pipeline:
        # Step 1: TF-IDF Vectorizer
        #   - ngram_range (1,2): unigrams AND bigrams (captures "invoice number", "due date")
        #   - max_features: vocabulary size limit
        #   - sublinear_tf: log(1+tf) — dampens effect of very frequent terms
        #   - min_df=1: include terms appearing at least once
        self.pipeline = Pipeline([
            ('tfidf', TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=5000,
                sublinear_tf=True,
                min_df=1,
                strip_accents='unicode',
                analyzer='word',
                token_pattern=r'\w{2,}'  # only words with 2+ chars
            )),
            # Step 2: Support Vector Machine (SVM)
            #   - kernel='rbf': Radial Basis Function — handles non-linear boundaries
            #   - C=5: regularization (higher = tighter fit, less regularization)
            #   - gamma='scale': auto-scale based on features
            #   - probability=True: enables predict_proba for confidence scores
            ('svm', SVC(
                kernel='rbf',
                C=5.0,
                gamma='scale',
                probability=True,
                class_weight='balanced',  # handles class imbalance
                random_state=42
            ))
        ])

        self.pipeline.fit(X, y)
        print(f"✅ DocumentClassifier trained on {len(X)} samples")

        # Save model
        os.makedirs(os.path.dirname(self.model_path), exist_ok=True)
        with open(self.model_path, 'wb') as f:
            pickle.dump(self.pipeline, f)

    def predict(self, text):
        """
        Classify a document text.
        Returns dict with: label, confidence, all_probabilities
        """
        if not text or len(text.strip()) < 10:
            return {
                'label'      : 'unknown',
                'confidence' : 0.0,
                'probabilities': {c: 0.0 for c in self.CLASSES}
            }

        # Clean text for classification
        text_lower = text.lower()

        # Get probabilities from SVM
        proba = self.pipeline.predict_proba([text_lower])[0]
        classes = self.pipeline.classes_

        # Map probabilities to class names
        prob_dict = {cls: float(p) for cls, p in zip(classes, proba)}

        # Predicted class = highest probability
        predicted_label = max(prob_dict, key=prob_dict.get)
        confidence = prob_dict[predicted_label]

        # ── Keyword override for high-confidence rule matches ─────────────────
        # If strong keywords present, boost confidence (hybrid approach)
        keyword_signals = {
            'invoice' : ['invoice', 'bill to', 'tax invoice', 'inv-', 'due date', 'gstin', 'unit price'],
            'contract': ['agreement', 'hereby', 'whereas', 'parties agree', 'governing law',
                        'termination', 'arbitration', 'indemnification', 'con-'],
            'report'  : ['executive summary', 'quarterly', 'annual report', 'kpi', 'ebitda',
                        'year over year', 'rpt-', 'financial report', 'audit'],
        }
        text_check = text_lower
        max_hits = 0
        keyword_winner = None
        for cls, keywords in keyword_signals.items():
            hits = sum(1 for kw in keywords if kw in text_check)
            if hits > max_hits:
                max_hits = hits
                keyword_winner = cls

        # If keyword signal strongly disagrees with SVM, blend
        if keyword_winner and keyword_winner != predicted_label and max_hits >= 3:
            prob_dict[keyword_winner] = min(0.95, prob_dict[keyword_winner] + 0.3 * max_hits / len(keyword_signals[keyword_winner]))
            predicted_label = max(prob_dict, key=prob_dict.get)
            confidence = prob_dict[predicted_label]

        return {
            'label'        : predicted_label,
            'confidence'   : round(confidence, 4),
            'probabilities': {k: round(v, 4) for k, v in prob_dict.items()}
        }

    def evaluate(self):
        """
        Evaluate classifier performance on a held-out test split.
        Returns metrics dict with accuracy, per-class precision/recall/F1.
        """
        X, y = self._build_training_set()
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )

        # Train fresh model on train split
        eval_pipeline = Pipeline([
            ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000,
                                      sublinear_tf=True, min_df=1, token_pattern=r'\w{2,}')),
            ('svm', SVC(kernel='rbf', C=5.0, gamma='scale', probability=True,
                        class_weight='balanced', random_state=42))
        ])
        eval_pipeline.fit(X_train, y_train)
        y_pred = eval_pipeline.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred,
                                       target_names=self.CLASSES,
                                       output_dict=True)
        cm = confusion_matrix(y_test, y_pred, labels=self.CLASSES)

        return {
            'accuracy'        : round(acc, 4),
            'classification_report': report,
            'confusion_matrix': cm.tolist(),
            'classes'         : self.CLASSES,
            'test_size'       : len(y_test),
            'train_size'      : len(X_train),
        }


# ─── Run standalone test ──────────────────────────────────────────────────────
if __name__ == "__main__":
    print("Testing FieldExtractor...")
    extractor = FieldExtractor()
    sample = """
    Invoice # INV-54321  Date: 15/06/2024  Due: 15/07/2024
    Bill To: TechVentures Pvt Ltd  GSTIN: 27ABCDE1234F1Z5
    Subtotal: $5,000.00  GST 18%: $900.00  TOTAL: $5,900.00
    """
    fields = extractor.extract(sample)
    print(f"  Dates     : {fields['dates']}")
    print(f"  Amounts   : {fields['amounts']}")
    print(f"  Doc IDs   : {fields['document_ids']}")
    print(f"  GST Nos   : {fields['gst_numbers']}")

    print("\nTesting DocumentClassifier...")
    classifier = DocumentClassifier()
    for text, expected in [
        ("Invoice INV-123 bill to client subtotal GST total amount due payment", "invoice"),
        ("Service agreement between parties governing law arbitration termination", "contract"),
        ("Quarterly report executive summary EBITDA KPI year over year growth", "report"),
    ]:
        result = classifier.predict(text)
        status = "✅" if result['label'] == expected else "❌"
        print(f"  {status} '{text[:50]}...' → {result['label']} ({result['confidence']:.1%})")

In [ ]:
"""
=============================================================
STEP 4: MODEL TESTING & EVALUATION
=============================================================
Validates and benchmarks the full pipeline:
  - OCR accuracy measurement
  - Classifier evaluation (accuracy, precision, recall, F1)
  - Confusion matrix generation
  - A/B testing across document conditions
  - End-to-end pipeline performance report
"""

import os
import sys
import json
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_fscore_support, accuracy_score
)

# Add project root to path
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..'))

from backend.nlp_extractor import DocumentClassifier, FieldExtractor
from backend.ocr_engine import OCREngine
from utils.document_generator import generate_all_samples
from utils.preprocess import DocumentPreprocessor

OUTPUTS_DIR = os.path.join(os.path.dirname(__file__), '..', 'outputs')
os.makedirs(OUTPUTS_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
class ModelEvaluator:
    """
    Comprehensive evaluator for the document processing pipeline.
    """

    def __init__(self):
        print("🔧 Initializing evaluator...")
        self.classifier    = DocumentClassifier()
        self.field_extractor = FieldExtractor()
        self.ocr_engine    = OCREngine()
        self.preprocessor  = DocumentPreprocessor()

    # ── 4A: Generate test documents ───────────────────────────────────────────
    def generate_test_set(self, n=5):
        """Generate labeled test documents."""
        print(f"\n📂 Generating {n*3} test documents...")
        docs = generate_all_samples(n_each=n)
        print(f"   ✅ {len(docs)} documents ready")
        return docs

    # ── 4B: Run full pipeline on one document ─────────────────────────────────
    def process_single(self, image_path, true_label=None):
        """
        Run the complete pipeline on one document:
        preprocess → OCR → classify → extract fields
        Returns result dict.
        """
        t_start = time.time()

        # OCR
        ocr_result = self.ocr_engine.process_document(image_path)
        text = ocr_result['clean_text']

        # Classify
        clf_result = self.classifier.predict(text)

        # Extract fields
        fields = self.field_extractor.extract(text)

        elapsed = time.time() - t_start

        return {
            'path'          : image_path,
            'true_label'    : true_label,
            'predicted_label': clf_result['label'],
            'confidence'    : clf_result['confidence'],
            'probabilities' : clf_result['probabilities'],
            'ocr_confidence': ocr_result['confidence'],
            'word_count'    : ocr_result['word_count'],
            'extracted_fields': fields,
            'clean_text'    : text,
            'processing_time': round(elapsed, 3),
            'correct'       : (clf_result['label'] == true_label) if true_label else None,
        }

    # ── 4C: Batch evaluation ───────────────────────────────────────────────────
    def evaluate_batch(self, docs):
        """
        Run pipeline on all test documents, collect metrics.
        """
        print(f"\n🔬 Running batch evaluation on {len(docs)} documents...")
        results = []

        for i, doc in enumerate(docs):
            print(f"   [{i+1}/{len(docs)}] {os.path.basename(doc['path'])} ({doc['type']})")
            result = self.process_single(doc['path'], true_label=doc['type'])
            results.append(result)

        return results

    # ── 4D: Compute metrics ────────────────────────────────────────────────────
    def compute_metrics(self, results):
        """
        Compute comprehensive classification metrics:
        - Accuracy, Precision, Recall, F1
        - Per-class breakdown
        - Confusion matrix
        """
        y_true = [r['true_label']    for r in results if r['true_label']]
        y_pred = [r['predicted_label'] for r in results if r['true_label']]

        if not y_true:
            return {}

        classes = ['invoice', 'contract', 'report']
        acc = accuracy_score(y_true, y_pred)
        prec, rec, f1, support = precision_recall_fscore_support(
            y_true, y_pred, labels=classes, zero_division=0
        )
        cm = confusion_matrix(y_true, y_pred, labels=classes)

        avg_conf     = np.mean([r['confidence'] for r in results])
        avg_ocr_conf = np.mean([r['ocr_confidence'] for r in results])
        avg_time     = np.mean([r['processing_time'] for r in results])

        per_class = {}
        for i, cls in enumerate(classes):
            per_class[cls] = {
                'precision': round(float(prec[i]), 4),
                'recall'   : round(float(rec[i]),  4),
                'f1_score' : round(float(f1[i]),   4),
                'support'  : int(support[i])
            }

        metrics = {
            'accuracy'       : round(float(acc), 4),
            'macro_precision': round(float(np.mean(prec)), 4),
            'macro_recall'   : round(float(np.mean(rec)),  4),
            'macro_f1'       : round(float(np.mean(f1)),   4),
            'per_class'      : per_class,
            'confusion_matrix': cm.tolist(),
            'classes'        : classes,
            'avg_clf_confidence': round(float(avg_conf), 4),
            'avg_ocr_confidence': round(float(avg_ocr_conf), 4),
            'avg_processing_time_s': round(float(avg_time), 3),
            'total_docs'     : len(results),
            'correct'        : sum(1 for r in results if r.get('correct')),
        }

        return metrics

    # ── 4E: Plot confusion matrix ──────────────────────────────────────────────
    def plot_confusion_matrix(self, metrics, save_path=None):
        """Generate and save confusion matrix heatmap."""
        cm = np.array(metrics['confusion_matrix'])
        classes = metrics['classes']

        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes,
            linewidths=0.5, linecolor='gray', ax=ax
        )
        ax.set_title('Confusion Matrix — Document Classifier', fontsize=14, fontweight='bold', pad=15)
        ax.set_xlabel('Predicted Label', fontsize=12)
        ax.set_ylabel('True Label', fontsize=12)

        # Add accuracy annotation
        ax.text(0.5, -0.12, f"Overall Accuracy: {metrics['accuracy']*100:.1f}%",
                transform=ax.transAxes, ha='center', fontsize=11,
                color='#1a3a6b', fontweight='bold')

        plt.tight_layout()
        path = save_path or os.path.join(OUTPUTS_DIR, 'confusion_matrix.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"   📊 Confusion matrix saved: {path}")
        return path

    # ── 4F: Plot per-class metrics bar chart ───────────────────────────────────
    def plot_metrics_chart(self, metrics, save_path=None):
        """Bar chart of precision, recall, F1 per class."""
        classes = metrics['classes']
        prec = [metrics['per_class'][c]['precision'] for c in classes]
        rec  = [metrics['per_class'][c]['recall']    for c in classes]
        f1   = [metrics['per_class'][c]['f1_score']  for c in classes]

        x = np.arange(len(classes))
        width = 0.25

        fig, ax = plt.subplots(figsize=(10, 6))
        colors = ['#1e3c7b', '#2e7bcf', '#5ba4f5']
        b1 = ax.bar(x - width, prec, width, label='Precision', color=colors[0], alpha=0.9)
        b2 = ax.bar(x,         rec,  width, label='Recall',    color=colors[1], alpha=0.9)
        b3 = ax.bar(x + width, f1,   width, label='F1-Score',  color=colors[2], alpha=0.9)

        # Annotate bars
        for bars in [b1, b2, b3]:
            for bar in bars:
                h = bar.get_height()
                ax.annotate(f'{h:.2f}',
                            xy=(bar.get_x() + bar.get_width()/2, h),
                            xytext=(0, 3), textcoords='offset points',
                            ha='center', va='bottom', fontsize=9)

        ax.set_ylim(0, 1.15)
        ax.set_title('Per-Class Classification Metrics', fontsize=14, fontweight='bold')
        ax.set_xlabel('Document Class', fontsize=12)
        ax.set_ylabel('Score', fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels([c.capitalize() for c in classes], fontsize=11)
        ax.legend(fontsize=10)
        ax.grid(axis='y', alpha=0.3)

        # Add macro averages text
        ax.text(0.98, 0.97,
                f"Macro F1: {metrics['macro_f1']:.3f}\n"
                f"Accuracy: {metrics['accuracy']*100:.1f}%",
                transform=ax.transAxes, ha='right', va='top',
                fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

        plt.tight_layout()
        path = save_path or os.path.join(OUTPUTS_DIR, 'metrics_chart.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"   📊 Metrics chart saved: {path}")
        return path

    # ── 4G: Plot processing time & confidence distribution ─────────────────────
    def plot_performance_overview(self, results, metrics, save_path=None):
        """Multi-panel overview: processing times, OCR confidence, classification confidence."""
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle('Pipeline Performance Overview', fontsize=14, fontweight='bold')

        # ① Processing time by document type
        times_by_type = {}
        for r in results:
            lbl = r.get('true_label', 'unknown')
            times_by_type.setdefault(lbl, []).append(r['processing_time'])

        types = list(times_by_type.keys())
        means = [np.mean(v) for v in times_by_type.values()]
        axes[0].bar(types, means, color=['#1e3c7b', '#2e7bcf', '#5ba4f5'], alpha=0.85)
        axes[0].set_title('Avg Processing Time (s)', fontweight='bold')
        axes[0].set_ylabel('Seconds')
        for i, m in enumerate(means):
            axes[0].text(i, m + 0.01, f'{m:.2f}s', ha='center', fontsize=9)
        axes[0].grid(axis='y', alpha=0.3)

        # ② OCR confidence distribution
        ocr_confs = [r['ocr_confidence'] for r in results]
        axes[1].hist(ocr_confs, bins=10, color='#2e7bcf', alpha=0.8, edgecolor='white')
        axes[1].axvline(np.mean(ocr_confs), color='red', linestyle='--',
                        label=f'Mean: {np.mean(ocr_confs):.1f}')
        axes[1].set_title('OCR Confidence Distribution', fontweight='bold')
        axes[1].set_xlabel('Confidence Score')
        axes[1].set_ylabel('Count')
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        # ③ Classification confidence pie
        correct   = sum(1 for r in results if r.get('correct') is True)
        incorrect = sum(1 for r in results if r.get('correct') is False)
        labels_pie = ['Correct', 'Incorrect']
        sizes = [correct, incorrect] if (correct + incorrect) > 0 else [1, 0]
        colors_pie = ['#2ecc71', '#e74c3c']
        axes[2].pie(sizes, labels=labels_pie, colors=colors_pie, autopct='%1.0f%%',
                    startangle=90, textprops={'fontsize': 11})
        axes[2].set_title(f'Classification Results\n(n={len(results)})', fontweight='bold')

        plt.tight_layout()
        path = save_path or os.path.join(OUTPUTS_DIR, 'performance_overview.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"   📊 Performance overview saved: {path}")
        return path

    # ── 4H: A/B Test — preprocessing ON vs OFF ────────────────────────────────
    def ab_test_preprocessing(self, docs, n_test=3):
        """
        Compare OCR accuracy with vs without preprocessing.
        Hypothesis: preprocessing improves OCR word count & confidence.
        """
        print(f"\n🔬 A/B Test: Preprocessing ON vs OFF (n={n_test})...")
        test_docs = docs[:n_test]

        results_with = []
        results_without = []

        for doc in test_docs:
            # WITH preprocessing (standard pipeline)
            r_with = self.ocr_engine.process_document(doc['path'])
            results_with.append({
                'confidence': r_with['confidence'],
                'words': r_with['word_count']
            })

            # WITHOUT preprocessing (raw Tesseract)
            r_without = self.ocr_engine.extract_text(doc['path'], preprocess=False)
            results_without.append({
                'confidence': r_without.get('confidence', 0),
                'words': r_without.get('word_count', 0)
            })

        avg_conf_with    = np.mean([r['confidence'] for r in results_with])
        avg_conf_without = np.mean([r['confidence'] for r in results_without])
        avg_words_with   = np.mean([r['words'] for r in results_with])
        avg_words_without= np.mean([r['words'] for r in results_without])

        ab_results = {
            'with_preprocessing': {
                'avg_confidence': round(float(avg_conf_with), 2),
                'avg_words': round(float(avg_words_with), 1)
            },
            'without_preprocessing': {
                'avg_confidence': round(float(avg_conf_without), 2),
                'avg_words': round(float(avg_words_without), 1)
            },
            'improvement_confidence': round(float(avg_conf_with - avg_conf_without), 2),
            'improvement_words': round(float(avg_words_with - avg_words_without), 1),
        }

        print(f"   WITH preprocessing    : conf={avg_conf_with:.1f}, words={avg_words_with:.0f}")
        print(f"   WITHOUT preprocessing : conf={avg_conf_without:.1f}, words={avg_words_without:.0f}")
        print(f"   Improvement           : conf +{ab_results['improvement_confidence']:.1f}, words +{ab_results['improvement_words']:.0f}")

        return ab_results

    # ── 4I: Save full evaluation report as JSON ────────────────────────────────
    def save_report(self, metrics, ab_results, results):
        """Save all evaluation data to JSON."""
        report = {
            'classification_metrics': metrics,
            'ab_test_results'        : ab_results,
            'individual_results'     : [
                {k: v for k, v in r.items() if k not in ('clean_text', 'extracted_fields')}
                for r in results
            ]
        }
        path = os.path.join(OUTPUTS_DIR, 'evaluation_report.json')
        with open(path, 'w') as f:
            json.dump(report, f, indent=2, default=str)
        print(f"   📄 Evaluation report saved: {path}")
        return path

    # ── MAIN RUN ───────────────────────────────────────────────────────────────
    def run_full_evaluation(self, n_docs=5):
        """Run the complete Step 4 evaluation pipeline."""
        print("\n" + "="*60)
        print("  STEP 4: MODEL TESTING & EVALUATION")
        print("="*60)

        # A: Generate test data
        docs = self.generate_test_set(n=n_docs)

        # B: Batch process
        results = self.evaluate_batch(docs)

        # C: Compute metrics
        metrics = self.compute_metrics(results)

        # D: Print summary
        print(f"\n📈 EVALUATION RESULTS")
        print(f"   Total Documents  : {metrics['total_docs']}")
        print(f"   Correct          : {metrics['correct']}/{metrics['total_docs']}")
        print(f"   Accuracy         : {metrics['accuracy']*100:.1f}%")
        print(f"   Macro Precision  : {metrics['macro_precision']:.3f}")
        print(f"   Macro Recall     : {metrics['macro_recall']:.3f}")
        print(f"   Macro F1         : {metrics['macro_f1']:.3f}")
        print(f"   Avg OCR Conf     : {metrics['avg_ocr_confidence']:.1f}%")
        print(f"   Avg Proc. Time   : {metrics['avg_processing_time_s']:.2f}s")
        print(f"\n   Per-class breakdown:")
        for cls, m in metrics['per_class'].items():
            print(f"     {cls:10s}: P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1_score']:.3f} (n={m['support']})")

        # E: Generate plots
        print("\n🎨 Generating evaluation charts...")
        self.plot_confusion_matrix(metrics)
        self.plot_metrics_chart(metrics)
        self.plot_performance_overview(results, metrics)

        # F: A/B test
        ab_results = self.ab_test_preprocessing(docs)

        # G: Save report
        self.save_report(metrics, ab_results, results)

        # H: Also evaluate the classifier internally
        print("\n🔬 Internal classifier cross-validation...")
        clf_eval = self.classifier.evaluate()
        print(f"   CV Accuracy: {clf_eval['accuracy']*100:.1f}%")
        print(f"   Train size : {clf_eval['train_size']}, Test size: {clf_eval['test_size']}")

        return metrics, results, ab_results


# ─── Run ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    evaluator = ModelEvaluator()
    metrics, results, ab = evaluator.run_full_evaluation(n_docs=5)
    print("\n✅ Evaluation complete. Check outputs/ directory for charts and report.")

In [ ]:
"""
=============================================================
STEP 5: FLASK BACKEND — REST API
=============================================================
Exposes the full document processing pipeline via HTTP API:
  POST /api/process    — upload & process a document image
  GET  /api/health     — health check
  GET  /api/metrics    — classifier evaluation metrics
"""

import os
import sys
import json
import base64
import tempfile
import traceback
from flask import Flask, request, jsonify
from werkzeug.utils import secure_filename
import numpy as np

# Add project root to path
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..'))

from backend.ocr_engine import OCREngine
from backend.nlp_extractor import DocumentClassifier, FieldExtractor
from utils.preprocess import DocumentPreprocessor

# ─── Flask App ────────────────────────────────────────────────────────────────
app = Flask(__name__)
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 16MB max upload

# ─── Lazy-loaded pipeline (initialized once on first request) ─────────────────
_pipeline = {}

def get_pipeline():
    """Initialize and cache the processing pipeline."""
    if not _pipeline:
        print("⚙️  Initializing pipeline components...")
        _pipeline['ocr']        = OCREngine()
        _pipeline['classifier'] = DocumentClassifier()
        _pipeline['extractor']  = FieldExtractor()
        _pipeline['preprocessor'] = DocumentPreprocessor()
        print("✅ Pipeline ready")
    return _pipeline

ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg', 'bmp', 'tiff', 'tif', 'webp'}

def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS


# ─── ROUTES ───────────────────────────────────────────────────────────────────

@app.route('/api/health', methods=['GET'])
def health():
    """Health check endpoint."""
    return jsonify({'status': 'ok', 'message': 'DocAI API is running'})


@app.route('/api/process', methods=['POST'])
def process_document():
    """
    Main processing endpoint.
    Accepts: multipart/form-data with 'file' field (image)
           OR JSON body with 'image_base64' field

    Returns JSON with:
      - document_type  : classified type
      - confidence     : classification confidence
      - extracted_text : OCR output
      - fields         : extracted structured fields
      - ocr_confidence : OCR accuracy
      - processing_time: seconds taken
    """
    import time
    t_start = time.time()
    pipe = get_pipeline()

    try:
        image_path = None

        # ── Accept file upload ─────────────────────────────────────────────────
        if 'file' in request.files:
            f = request.files['file']
            if f.filename == '':
                return jsonify({'error': 'No file selected'}), 400
            if not allowed_file(f.filename):
                return jsonify({'error': f'File type not allowed. Use: {ALLOWED_EXTENSIONS}'}), 400

            # Save to temp file
            suffix = '.' + f.filename.rsplit('.', 1)[1].lower()
            with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
                f.save(tmp.name)
                image_path = tmp.name

        # ── Accept base64 encoded image ────────────────────────────────────────
        elif request.is_json and 'image_base64' in request.json:
            b64_data = request.json['image_base64']
            # Strip data URL prefix if present
            if ',' in b64_data:
                b64_data = b64_data.split(',')[1]
            img_bytes = base64.b64decode(b64_data)
            ext = request.json.get('format', 'png')
            with tempfile.NamedTemporaryFile(suffix=f'.{ext}', delete=False) as tmp:
                tmp.write(img_bytes)
                image_path = tmp.name

        else:
            return jsonify({'error': 'No file or image_base64 provided'}), 400

        # ── Run full pipeline ──────────────────────────────────────────────────
        # 1. OCR
        ocr_result = pipe['ocr'].process_document(image_path)
        text = ocr_result['clean_text']

        # 2. Classify
        clf = pipe['classifier'].predict(text)

        # 3. Extract fields
        fields = pipe['extractor'].extract(text)

        # Clean up temp file
        try:
            os.unlink(image_path)
        except:
            pass

        elapsed = round(time.time() - t_start, 3)

        # ── Serialize fields (remove non-JSON-safe values) ─────────────────────
        def safe(v):
            if isinstance(v, (np.integer, np.floating)):
                return float(v)
            if isinstance(v, np.ndarray):
                return v.tolist()
            return v

        safe_fields = {k: safe(v) for k, v in fields.items() if k != 'summary'}

        return jsonify({
            'success'           : True,
            'document_type'     : clf['label'],
            'confidence'        : clf['confidence'],
            'type_probabilities': clf['probabilities'],
            'ocr_confidence'    : ocr_result['confidence'],
            'word_count'        : ocr_result['word_count'],
            'extracted_text'    : text[:3000],  # Limit response size
            'fields'            : safe_fields,
            'field_summary'     : fields['summary'],
            'processing_time_s' : elapsed,
        })

    except Exception as e:
        # Clean up temp file on error
        if image_path and os.path.exists(image_path):
            try:
                os.unlink(image_path)
            except:
                pass
        return jsonify({
            'success': False,
            'error': str(e),
            'trace': traceback.format_exc()
        }), 500


@app.route('/api/metrics', methods=['GET'])
def get_metrics():
    """Return classifier evaluation metrics."""
    pipe = get_pipeline()
    try:
        metrics = pipe['classifier'].evaluate()
        return jsonify({'success': True, 'metrics': metrics})
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500


@app.route('/api/classify-text', methods=['POST'])
def classify_text():
    """Classify raw text (no OCR step)."""
    pipe = get_pipeline()
    data = request.get_json()
    if not data or 'text' not in data:
        return jsonify({'error': 'Missing text field'}), 400

    text   = data['text']
    clf    = pipe['classifier'].predict(text)
    fields = pipe['extractor'].extract(text)

    return jsonify({
        'document_type'     : clf['label'],
        'confidence'        : clf['confidence'],
        'type_probabilities': clf['probabilities'],
        'fields'            : {k: v for k, v in fields.items() if k != 'summary'},
        'field_summary'     : fields['summary'],
    })


# ─── Run ──────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    print("🚀 Starting DocAI Flask API...")
    get_pipeline()
    app.run(host='0.0.0.0', port=5001, debug=False)